Prédiction en temps réel (simulé) sur Silver Streaming avec ton modèle MLflow

In [0]:
import mlflow
import mlflow.spark
import os

# Obligatoire en Free Edition pour SparkML + UC
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/main/ml/models_volume/tmp"
dbutils.fs.mkdirs("/Volumes/main/ml/models_volume/tmp")

# Charger le modèle depuis Unity Catalog
model_uri = "models:/main.ml.fraud_detection_model/2"
# Pour production, créez un alias "champion" et utilisez: models:/main.ml.fraud_detection_model@champion

model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/main/ml/models_volume/tmp"
)

print("Modèle MLflow chargé avec succès :", model_uri)


In [0]:
# Lecture du flux Silver Streaming
silver_stream = (
    spark.readStream
         .table("main.silver.transactions_silver_stream")
)


In [0]:
silver_stream.printSchema()

In [0]:
# Préparation des features avec VectorAssembler
from pyspark.ml.feature import VectorAssembler

feature_cols = [f"V{i}" for i in range(1, 29)] + ["amount"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

stream_features = assembler.transform(silver_stream)


In [0]:
stream_features

In [0]:
# Prédiction sur le flux Silver Streaming
pred_stream = model.transform(stream_features)

# Choix des colonnes pour la table de prédiction
predictions = (
    pred_stream
    .select(
        "time",
        "amount",
        "prediction",
        "probability",
        "is_fraud"
    )
)


In [0]:
# Ecriture en streaming dans une table Delta Gold
predictions.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/main/gold/gold_volume/_checkpoints/predictions_stream") \
    .trigger(once=True) \
    .table("main.gold.fraud_predictions_stream")
